In [ ]:
# Import packages

import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import xarray
import math
import glob
import os
import io
import pathlib
import requests

import rasterio
from rasterio.merge import merge
from rasterio.plot import show
from rasterio.mask import mask
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling
from rasterio.crs import CRS
from rasterio.features import rasterize

import shapely
from shapely.geometry import Polygon
from shapely.geometry import mapping
from shapely.geometry import box

from tqdm import tqdm
from affine import Affine
from zipfile import ZipFile

import logging
logging.basicConfig(level=logging.INFO)

import atlite
from atlite.gis import ExclusionContainer, shape_availability

In [ ]:
aggregated_regions = [
    "BE", "DE",
    "DK", "FR", "UK",
    "IE", 
    "LU", "NL", "NO", 
]

In [ ]:
europe = (
    gpd
    .read_file(snakemake.input.euroshape)
    .query("NUTS_ID == @aggregated_regions")
    .set_index(["NUTS_ID"])
    .loc[:,['geometry']]
)

## Functions

In [ ]:
def plot_eligible_area(ax, tiff_paths, europe, title):
    excluder_wind_onshore = ExclusionContainer()

    full_europe = (
        europe
        .assign(col='a')
        .dissolve(by='col')
        .geometry
    )

    full_europe = full_europe.to_crs(excluder_wind_onshore.crs)
    for tiff_path in tiff_paths:
        excluder_wind_onshore.add_raster(tiff_path)
    masked, transform = shape_availability(full_europe, excluder_wind_onshore)
    eligible_share = (masked.sum() * excluder_wind_onshore.res**2 / full_europe.geometry.item().area)
    
    # Plot the eligible area in a subplot
    show(masked, transform=transform, cmap='Greens', ax=ax)
    full_europe.plot(ax=ax, edgecolor='k', color='None')
    europe.to_crs(excluder_wind_onshore.crs).boundary.plot(ax=ax, edgecolor='grey', linewidth=0.2)
    ax.set_title(f'{title} \n Eligible area (green) {eligible_share * 100:2.2f}%')

In [ ]:
def get_bounding():
    
    rectx1 = -12
    rectx2 = 44
    recty1 = 33
    recty2 = 72
    
    polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
    )
    
    polygon=shapely.segmentize(polygon, max_segment_length=0.5)
    
    b=gpd.GeoDataFrame(geometry=[polygon],crs="EPSG:4326")
 
    return b.to_crs("EPSG:3035")

In [ ]:
from pyproj import Transformer

rectx1 = -12
rectx2 = 44
recty1 = 33
recty2 = 72

# Define the transformer to convert from WGS84 to EPSG:3035
transformer = Transformer.from_crs("epsg:4326", "epsg:3035", always_xy=True)

# Example conversion
lon, lat = rectx1, recty1  # Example coordinates in WGS84
x, y = transformer.transform(lon, lat)
print(f"Projected coordinates: {x}, {y}")

In [ ]:
    
rectx1 = 2268030.888058131
rectx2 = 5453593.761521462
recty1 = 1401640.028262999
recty2 = 5718178.337705897

polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
)
europe = europe.to_crs(get_bounding().crs)
europe = gpd.clip(europe, polygon)

In [ ]:
pipelines = gpd.read_file(snakemake.input.pipelinesPath).to_crs(get_bounding().crs)
pipelines = gpd.clip(pipelines, polygon)
fixed_platforms = gpd.read_file(snakemake.input.fixed_platformsPath).to_crs(get_bounding().crs)
fixed_platforms = gpd.clip(fixed_platforms, polygon)
floating_platforms = gpd.read_file(snakemake.input.float_platformsPath).to_crs(get_bounding().crs)
floating_platforms = gpd.clip(floating_platforms, polygon)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(20,10))

europe.boundary.plot(ax=axes[0],linewidth=0.5,)
pipelines.plot(ax=axes[0], color='green',linewidth=0.5,alpha=0.5)
fixed_platforms.plot(ax=axes[0], color ='blue',markersize=1)
floating_platforms.plot(ax=axes[0], color = 'red', markersize=1)

europe.boundary.plot(ax=axes[1],linewidth=0.5,)
#pipelines.plot(ax=axes[1], color='green',linewidth=0.5,alpha=0.5)

buffered_pipes = pipelines.copy()
buffered_pipes['geometry'] = buffered_pipes['geometry'].buffer(snakemake.params.offshoreBuffer.get('pipelines'))
buffered_fixed = fixed_platforms.copy()
buffered_fixed['geometry'] = buffered_fixed['geometry'].buffer(snakemake.params.offshoreBuffer.get('fixed_platforms'))
buffered_float = floating_platforms.copy()
buffered_float['geometry'] = buffered_float['geometry'].buffer(snakemake.params.offshoreBuffer.get('float_platforms'))

buffered_fixed.plot(ax=axes[1], color ='blue')
buffered_float.plot(ax=axes[1], color ='red')
buffered_pipes.plot(ax=axes[1], color ='green')

In [ ]:
import rasterio
from rasterio.merge import merge
from rasterio.plot import show
from rasterio.mask import mask
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling
from rasterio.crs import CRS
from rasterio.features import rasterize

In [ ]:
snakemake.output.bufferedPipes

In [ ]:
minx, miny, maxx, maxy = get_bounding().total_bounds
resolution = 100

gdf = buffered_pipes

# Generate a mask for the rasterization
transform = from_bounds(minx, miny, maxx, maxy, int((maxx - minx) / resolution), int((maxy - miny) / resolution))

# Rasterize the GeoDataFrame
out_shape = (int((maxy - miny) / resolution), int((maxx - minx) / resolution))
rasterized = rasterize(
    [(geom, 1) for geom in gdf.geometry], 
    out_shape=out_shape,
    transform=transform,
    fill=0,
    all_touched=True,
    dtype='uint8'
)

# Define the raster metadata
raster_meta = {
    'driver': 'GTiff',
    'height': out_shape[0],
    'width': out_shape[1],
    'count': 1,
    'dtype': 'uint8',
    'crs': 'EPSG:3035',
    'transform': transform,
    'compress': 'lzw'  # Apply compression
}

output_raster = snakemake.output.bufferedPipes

with rasterio.open(output_raster, 'w', **raster_meta) as dst:
    dst.write(rasterized, 1)

print(f"Raster saved to {output_raster}")

In [ ]:
minx, miny, maxx, maxy = get_bounding().total_bounds
resolution = 100

gdf = buffered_float

# Generate a mask for the rasterization
transform = from_bounds(minx, miny, maxx, maxy, int((maxx - minx) / resolution), int((maxy - miny) / resolution))

# Rasterize the GeoDataFrame
out_shape = (int((maxy - miny) / resolution), int((maxx - minx) / resolution))
rasterized = rasterize(
    [(geom, 1) for geom in gdf.geometry], 
    out_shape=out_shape,
    transform=transform,
    fill=0,
    all_touched=True,
    dtype='uint8'
)

# Define the raster metadata
raster_meta = {
    'driver': 'GTiff',
    'height': out_shape[0],
    'width': out_shape[1],
    'count': 1,
    'dtype': 'uint8',
    'crs': 'EPSG:3035',
    'transform': transform,
    'compress': 'lzw'  # Apply compression
}

output_raster = snakemake.output.bufferedFloat

with rasterio.open(output_raster, 'w', **raster_meta) as dst:
    dst.write(rasterized, 1)

print(f"Raster saved to {output_raster}")

In [ ]:
minx, miny, maxx, maxy = get_bounding().total_bounds
resolution = 100

gdf = buffered_fixed

# Generate a mask for the rasterization
transform = from_bounds(minx, miny, maxx, maxy, int((maxx - minx) / resolution), int((maxy - miny) / resolution))

# Rasterize the GeoDataFrame
out_shape = (int((maxy - miny) / resolution), int((maxx - minx) / resolution))
rasterized = rasterize(
    [(geom, 1) for geom in gdf.geometry], 
    out_shape=out_shape,
    transform=transform,
    fill=0,
    all_touched=True,
    dtype='uint8'
)

# Define the raster metadata
raster_meta = {
    'driver': 'GTiff',
    'height': out_shape[0],
    'width': out_shape[1],
    'count': 1,
    'dtype': 'uint8',
    'crs': 'EPSG:3035',
    'transform': transform,
    'compress': 'lzw'  # Apply compression
}

output_raster = snakemake.output.bufferedFixed

with rasterio.open(output_raster, 'w', **raster_meta) as dst:
    dst.write(rasterized, 1)

print(f"Raster saved to {output_raster}")